# Heckman's Sample-Selection Model (Heckit)

When the outcome of interest is observed only for individuals who 'self-select' into the sample (think wages — observed only for those who work) and that selection is correlated with the outcome error, OLS on the selected subsample is biased. Heckman (1979) gives a two-step correction; the joint maximum-likelihood estimator is the asymptotically efficient alternative.

Model:

$$Y^* = X'\beta + e, \quad S^* = Z'\gamma + u, \quad S = \mathbf 1\{S^* > 0\},$$
$$Y = Y^* \text{ if } S = 1, \text{ missing otherwise}, \quad (e, u) \sim \mathcal N\!\left(0, \begin{pmatrix}\sigma^2 & \rho\sigma \\ \rho\sigma & 1\end{pmatrix}\right).$$

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from censtrunc import HeckitRegression

rng = np.random.default_rng(42)
n = 4000
# Shared regressor (in both equations); two exclusive regressors
shared  = rng.normal(size=n)
x_only  = rng.normal(size=n)       # in the outcome eq. only
z_only  = rng.normal(size=n)       # in the selection eq. only (exclusion)

beta_true  = np.array([1.0,  0.5, -0.3])     # const, shared, x_only
gamma_true = np.array([0.0,  0.3,  0.6])     # const, shared, z_only
rho_true, sigma_true = 0.6, 1.0

Sigma = np.array([[sigma_true**2, rho_true*sigma_true],
                  [rho_true*sigma_true, 1.0]])
errs = rng.multivariate_normal([0.0, 0.0], Sigma, size=n)
e, u = errs[:, 0], errs[:, 1]

X = np.column_stack([shared, x_only])
Z = np.column_stack([shared, z_only])
S = (gamma_true[0] + Z @ gamma_true[1:] + u > 0).astype(int)
y_full = beta_true[0] + X @ beta_true[1:] + e
y = np.where(S == 1, y_full, np.nan)
print(f'Selected (S=1): {S.sum()} / {n}  ({S.mean():.1%})')

Selected (S=1): 2022 / 4000  (50.5%)


## Why OLS on the selected sample is biased

Because $u$ and $e$ are correlated ($\rho > 0$), individuals with high $e$ tend to have high $u$ and therefore are more likely to be selected. The conditional mean for the selected sample is

$$\mathbb E[Y \mid X, S = 1] = X'\beta + \rho\sigma\,\lambda(Z'\gamma),$$

with $\lambda(\cdot)$ the inverse Mills ratio. Omitting $\lambda(Z'\gamma)$ from the regression — what naive OLS does — produces omitted-variable bias on $\beta$.

In [2]:
sel = ~np.isnan(y)
X_full = sm.add_constant(X)
ols = sm.OLS(y[sel], X_full[sel]).fit()
print('OLS on selected subsample (biased):')
print(np.asarray(ols.params).round(4))
print('truth:', beta_true)

OLS on selected subsample (biased):
[ 1.3651  0.4366 -0.3037]
truth: [ 1.   0.5 -0.3]


## Two-step Heckit

Heckman's two-step recipe:

1. Probit of $S$ on $Z$ to obtain $\hat\gamma$.
2. Compute the inverse Mills ratio $\hat\lambda_i = \phi(Z_i'\hat\gamma)/\Phi(Z_i'\hat\gamma)$ for selected observations.
3. OLS of $y_i$ on $(X_i,\hat\lambda_i)$ for selected $i$. The coefficients are $\hat\beta$ and $\hat\rho\hat\sigma$.

In [3]:
m_two = HeckitRegression(method='twostep').fit(y, X, Z)
print(m_two.summary())

                         Heckman Selection Regression                         
Method:                  twostep                  No. Observations:        4000
Selected (S=1):          2022                     Censored (S=0):          1978
sigma_e:                 0.952857                 rho:                     0.531381
                     Outcome equation (Y* = X * beta + e)                     
------------------------------------------------------------------------------
                     coef     std err         z     P>|z|      [0.025    0.975]
------------------------------------------------------------------------------
const              1.0267      0.0491   20.9278    0.0000      0.9306    1.1229
x1                 0.4903      0.0223   21.9471    0.0000      0.4465    0.5341
x2                -0.3010      0.0209  -14.4109    0.0000     -0.3419   -0.2601
------------------------------------------------------------------------------
             Selection equation (S* = Z *

## Joint MLE Heckit

Maximise the full log-likelihood (Hansen 2022, §27.10):

$$\ell = \sum_{S_i=0} \log[1 - \Phi(Z_i'\gamma)] + \sum_{S_i=1}\!\left\{\log\Phi\!\left(\frac{Z_i'\gamma + (\rho/\sigma)(Y_i - X_i'\beta)}{\sqrt{1 - \rho^2}}\right) - \tfrac12\log(2\pi\sigma^2) - \frac{(Y_i - X_i'\beta)^2}{2\sigma^2}\right\}.$$

Asymptotically efficient, somewhat slower than the two-step.

In [4]:
m_ml = HeckitRegression(method='mle').fit(y, X, Z)
print(m_ml.summary())

                         Heckman Selection Regression                         
Method:                  mle                      No. Observations:        4000
Selected (S=1):          2022                     Censored (S=0):          1978
sigma_e:                 0.958607                 rho:                     0.552343
Log-Likelihood:          -4924.8374
                     Outcome equation (Y* = X * beta + e)                     
------------------------------------------------------------------------------
                     coef     std err         z     P>|z|      [0.025    0.975]
------------------------------------------------------------------------------
const              1.0108      0.0446   22.6544    0.0000      0.9234    1.0983
x1                 0.4925      0.0214   23.0340    0.0000      0.4506    0.5344
x2                -0.3017      0.0192  -15.7382    0.0000     -0.3393   -0.2642
------------------------------------------------------------------------------
     

## Side-by-side comparison

All three estimators on the same data. Heckit's two-step and MLE recover $\beta_{\text{shared}}$ much closer to the truth than OLS, and they recover $\rho$ and $\sigma$ as well.

In [5]:
comparison = pd.DataFrame({
    'true':         beta_true,
    'OLS (biased)': np.asarray(ols.params),
    'Heckit 2step': m_two.coef_,
    'Heckit MLE':   m_ml.coef_,
}, index=['const', 'shared', 'x_only'])
comparison['OLS error']   = comparison['OLS (biased)'] - comparison['true']
comparison['2step error'] = comparison['Heckit 2step']  - comparison['true']
comparison['MLE error']   = comparison['Heckit MLE']    - comparison['true']
comparison.round(4)

,true,OLS (biased),Heckit 2step,Heckit MLE,OLS error,2step error,MLE error
const,1.0,1.3651,1.0267,1.0108,0.3651,0.0267,0.0108
shared,0.5,0.4366,0.4903,0.4925,-0.0634,-0.0097,-0.0075
x_only,-0.3,-0.3037,-0.3010,-0.3017,-0.0037,-0.0010,-0.0017


In [6]:
pd.DataFrame({
    'true':      [rho_true, sigma_true],
    'two-step':  [m_two.rho_, m_two.sigma_],
    'MLE':       [m_ml.rho_,  m_ml.sigma_],
}, index=['rho', 'sigma']).round(4)

,true,two-step,MLE
rho,0.6,0.5314,0.5523
sigma,1.0,0.9529,0.9586


## Cross-validation against `py4etrics` and R's `sampleSelection`

Recovering the true DGP is a necessary but not sufficient check — a buggy estimator could return sensible-looking numbers and still pass it. The decisive test is replication of an *independent* implementation on the *same* data. We compare against two:

- **`py4etrics.Heckit`** (Hasebe et al.) — a Python implementation of the closed-form two-step Heckman estimator.
- **`sampleSelection::selection`** (Toomet & Henningsen, JSS 27(7), 2008) — the standard Heckit package in R, with both estimators (two-step and joint MLE) implemented on the same log-likelihood we use.

Both checks are run as part of the test suite
(`tests/test_heckit.py::test_twostep_matches_py4etrics` and
`tests/test_r_reference.py::test_heckit_mle_matches_R`). The cells below reproduce them live.

If `sampleSelection` is not installed locally, on macOS Apple Silicon run `brew install nlopt cmake pkg-config` once and then `install.packages('sampleSelection', dependencies = TRUE)`.

In [7]:
# Compare with py4etrics (two-step only -- py4etrics MLE is a no-op).
import warnings
try:
    from py4etrics.heckit import Heckit as Py4Heckit
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        ref = Py4Heckit(y, sm.add_constant(X), sm.add_constant(Z)).fit(method='twostep')
    diff_beta  = np.max(np.abs(m_two.coef_  - ref.params))
    diff_gamma = np.max(np.abs(m_two.gamma_ - ref.select_res.params))
    print(f'max |beta_censtrunc  - beta_py4etrics|  = {diff_beta:.2e}')
    print(f'max |gamma_censtrunc - gamma_py4etrics| = {diff_gamma:.2e}')
    print(f'|sigma_censtrunc - sigma_py4etrics|     = {abs(m_two.sigma_ - np.sqrt(ref.var_reg_error)):.2e}')
    print(f'|rho_censtrunc   - rho_py4etrics|       = {abs(m_two.rho_   - ref.corr_eqnerrors):.2e}')
except ImportError:
    print('py4etrics not installed; pip install py4etrics to enable the comparison.')

max |beta_censtrunc  - beta_py4etrics|  = 1.88e-09
max |gamma_censtrunc - gamma_py4etrics| = 4.68e-09
|sigma_censtrunc - sigma_py4etrics|     = 1.08e-09
|rho_censtrunc   - rho_py4etrics|       = 3.16e-09


Both packages solve the same closed-form two-step expressions, so the residual disagreement (around $10^{-6}$) is just numpy / statsmodels linear-algebra round-off.

In [8]:
# Compare with R's sampleSelection::selection (joint MLE).
# The R script expects a CSV with columns (y, s, x1, x2, z1, z2) and fits
# outcome 'y ~ x1 + x2' / selection 's ~ x1 + x2 + z1 + z2', so we build
# a fresh DGP with that exact layout.
import shutil, subprocess, json, tempfile, os
from pathlib import Path

def _find_r_heckit():
    for c in [Path('tests/reference/fit_heckit.R'),
              Path('../tests/reference/fit_heckit.R')]:
        if c.exists():
            return c
    return None

rscript, rpath = shutil.which('Rscript'), _find_r_heckit()
if rscript and rpath:
    rng_r = np.random.default_rng(20250601)
    n_r = 4000
    x1r = rng_r.normal(size=n_r); x2r = rng_r.normal(size=n_r)
    z1r = rng_r.normal(size=n_r); z2r = rng_r.normal(size=n_r)
    bb = np.array([0.5, 1.0, -0.4])
    gg = np.array([0.1, 0.3, -0.2, 0.5, 0.7])
    rho_r, sigma_r = 0.5, 1.2
    Sig = np.array([[sigma_r**2, rho_r*sigma_r],[rho_r*sigma_r, 1.0]])
    er, ur = rng_r.multivariate_normal([0, 0], Sig, size=n_r).T
    y_lat = bb[0] + bb[1]*x1r + bb[2]*x2r + er
    s_lat = gg[0] + gg[1]*x1r + gg[2]*x2r + gg[3]*z1r + gg[4]*z2r + ur
    s_obs = (s_lat > 0).astype(int)
    y_obs_r = np.where(s_obs == 1, y_lat, np.nan)

    Xr = np.column_stack([x1r, x2r])
    Zr = np.column_stack([x1r, x2r, z1r, z2r])
    m_ml_r = HeckitRegression(method='mle').fit(y_obs_r, Xr, Zr)

    fd, csvp = tempfile.mkstemp(suffix='.csv'); os.close(fd)
    pd.DataFrame({
        'y': y_obs_r, 's': s_obs,
        'x1': x1r, 'x2': x2r, 'z1': z1r, 'z2': z2r,
    }).to_csv(csvp, index=False, na_rep='NA')
    try:
        out = subprocess.run([rscript, str(rpath), csvp],
                             capture_output=True, text=True, timeout=180, check=True)
        rdat = json.loads(out.stdout)['mle']
        r_beta  = np.array([rdat['outcome']['(Intercept)'],
                            rdat['outcome']['x1'], rdat['outcome']['x2']])
        r_gamma = np.array([rdat['selection']['(Intercept)'],
                            rdat['selection']['x1'], rdat['selection']['x2'],
                            rdat['selection']['z1'], rdat['selection']['z2']])
        print(f"max |beta_censtrunc  - beta_R|   = {np.max(np.abs(m_ml_r.coef_  - r_beta)):.2e}")
        print(f"max |gamma_censtrunc - gamma_R|  = {np.max(np.abs(m_ml_r.gamma_ - r_gamma)):.2e}")
        print(f"|sigma_censtrunc - sigma_R|      = {abs(m_ml_r.sigma_ - rdat['sigma']):.2e}")
        print(f"|rho_censtrunc   - rho_R|        = {abs(m_ml_r.rho_   - rdat['rho']):.2e}")
        print(f"|logLik_censtrunc - logLik_R|    = {abs(m_ml_r.llf_   - rdat['logLik']):.2e}")
    except Exception as exc:
        print(f'R run failed: {exc!r}')
    finally:
        os.remove(csvp)
else:
    print('Rscript not available at build time; see tests/test_r_reference.py for the gated test.')

max |beta_censtrunc  - beta_R|   = 5.68e-06
max |gamma_censtrunc - gamma_R|  = 6.81e-06
|sigma_censtrunc - sigma_R|      = 1.62e-06
|rho_censtrunc   - rho_R|        = 4.53e-06
|logLik_censtrunc - logLik_R|    = 9.93e-08


Two independent optimisers (scipy `L-BFGS-B` and R's `maxLik`) on the same joint log-likelihood: parameters, $\sigma$, $\rho$ and the log-likelihood all match to optimiser tolerance (~$10^{-3}$).

## Six kinds of prediction

Six Heckman quantities are exposed through a single `predict()`. Each has a one-letter code so that any subset can be requested at once. With $\lambda(t) = \phi(t)/\Phi(t)$ the inverse Mills ratio:

**Selection equation** (uses $Z$):

- **`s`** — selection probability: $P(S=1 \mid Z) = \Phi(Z'\hat\gamma)$
- **`n`** — non-selection probability: $P(S=0 \mid Z) = 1 - \Phi(Z'\hat\gamma)$
- **`p`** — selection propensity (latent index): $Z'\hat\gamma$

**Outcome equation** (uses $X$, sometimes both):

- **`o`** — $\mathbb E[Y \mid X, Z, S=1] = X'\hat\beta + \hat\rho\hat\sigma\,\lambda(Z'\hat\gamma)$ (observed conditional)
- **`h`** — $\mathbb E[Y^* \mid X] = X'\hat\beta$ (hidden / unconditional latent)
- **`u`** — $\mathbb E[Y^* \mid X, Z, S=0] = X'\hat\beta - \hat\rho\hat\sigma\,\phi(Z'\hat\gamma)/[1-\Phi(Z'\hat\gamma)]$ (unobserved conditional)

Calling `predict(X=..., Z=...)` with no `kind` returns all six columns as a `DataFrame` in the order `s, n, p, o, h, u`. Subsets are selected by letter string (`kind='sn'`, `kind='ohu'`, ...). A single letter returns a 1-D `ndarray`. Long names `'outcome'`, `'selection_prob'`, `'conditional'` keep working for backward compatibility (they alias `'h'`, `'s'`, `'o'`).

In [9]:
preds = m_ml.predict(X=X[:6], Z=Z[:6])   # default = all six
preds.assign(observed_y=y[:6], selected=S[:6]).round(3)

,prob_selected,prob_not_selected,propensity,observed,hidden,unobserved,observed_y,selected
0,0.626,0.374,0.322,1.405,1.084,0.548,1.327,1
1,0.704,0.296,0.536,0.488,0.229,-0.390,1.942,1
2,0.839,0.161,0.988,1.453,1.298,0.495,1.469,1
3,0.499,0.501,-0.003,1.222,0.798,0.377,1.666,1
4,0.363,0.637,-0.351,0.166,-0.381,-0.693,-0.354,1
5,0.071,0.929,-1.467,1.474,0.462,0.385,NaN,0


Subsets via letter strings. The two probabilities sum to one; the three E[Y]-style columns satisfy $\mathbb E[Y\cdot S] = \Phi(Z'\gamma) \cdot o + [1-\Phi(Z'\gamma)] \cdot 0$, which is just $\Phi(Z'\gamma) \cdot o$ on the selected and 0 on the unselected when the latter contribute $Y = 0$.

In [10]:
m_ml.predict(Z=Z[:6], kind='sn').round(3)   # only the two probabilities

,prob_selected,prob_not_selected
0,0.626,0.374
1,0.704,0.296
2,0.839,0.161
3,0.499,0.501
4,0.363,0.637
5,0.071,0.929


In [11]:
m_ml.predict(X=X[:6], Z=Z[:6], kind='ohu').round(3)   # the three E[Y] variants

,observed,hidden,unobserved
0,1.405,1.084,0.548
1,0.488,0.229,-0.390
2,1.453,1.298,0.495
3,1.222,0.798,0.377
4,0.166,-0.381,-0.693
5,1.474,0.462,0.385


In [12]:
m_ml.predict(X=X[:6], kind='h').round(3)   # single letter -> 1-D ndarray

array([ 1.084,  0.229,  1.298,  0.798, -0.381,  0.462])

## Same fit via a patsy formula

Instead of building `y`, `X`, `Z` by hand, every model class accepts a `from_formula(...)` constructor in the statsmodels style. For Heckit you pass *two* formulas — outcome and selection — sharing a single dataframe:

In [13]:
df = pd.DataFrame({
    'shared': shared, 'x_only': x_only, 'z_only': z_only,
    'inlf':   S, 'lwage': y_full,
})
m_form = HeckitRegression.from_formula(
    outcome   = 'lwage ~ 1 + shared + x_only',
    selection = 'inlf  ~ 1 + shared + z_only',
    data=df, method='mle',
).fit()
print(pd.DataFrame({
    'beta (explicit)': m_ml.coef_,
    'beta (formula)':  m_form.coef_,
}, index=['const', 'shared', 'x_only']).round(6))

        beta (explicit)  beta (formula)
const          1.010834        1.010834
shared         0.492478        0.492478
x_only        -0.301747       -0.301747


## Bootstrap standard errors

Two-step second-stage standard errors are naive: they ignore the variability of $\hat\gamma$ from the probit step. A paired bootstrap gives proper SEs. The MLE Hessian-based SEs are already correct.

In [14]:
boot = m_two.bootstrap(y, X, Z, n_boot=80, seed=0)
pd.DataFrame({
    'beta':         m_two.coef_,
    'naive SE':     m_two.bse_,
    'bootstrap SE': boot['beta_se'],
}, index=['const', 'shared', 'x_only']).round(4)

,beta,naive SE,bootstrap SE
const,1.0267,0.0491,0.0479
shared,0.4903,0.0223,0.0214
x_only,-0.3010,0.0209,0.0205


## Marginal effects

Heckman's model has four standard marginal-effect quantities (Greene, *Econometric Analysis* §19.5):

| `kind`            | Differentiates                                                         |
|-------------------|------------------------------------------------------------------------|
| `'latent'`        | $E[Y^*\mid X] = X'\beta$ — X-side only                                |
| `'conditional'`   | $E[Y\mid X, Z, S=1] = X'\beta + \rho\sigma\,\lambda(Z'\gamma)$        |
| `'unconditional'` | $E[Y\cdot S\mid X, Z] = \Phi(Z'\gamma)X'\beta + \rho\sigma\,\phi(Z'\gamma)$ |
| `'prob-selected'` | $P(S=1\mid Z) = \Phi(Z'\gamma)$ — Z-side only                          |

A variable that appears in *both* equations (here `shared`) has its X- and Z-side derivatives **added together**. SEs come from the delta method on the joint MLE covariance; for a two-step fit they are `NaN` because the joint covariance is not available in closed form (use bootstrap instead).

Closed-form per-row derivatives with $\lambda = \phi/\Phi$ and $\delta(t) = \lambda(t)(t + \lambda(t))$ (so $\mathrm d\lambda/\mathrm dt = -\delta$):

$$\frac{\partial E[Y\mid S=1]}{\partial v} = \beta_v\,\mathbf 1\{v\in X\} \;-\; \gamma_v\,\rho\sigma\,\delta(Z'\gamma)\,\mathbf 1\{v\in Z\}.$$

We pass `feature_names_outcome` / `feature_names_selection` so that variables with matching names in $X$ and $Z$ are recognised as the same variable.

In [15]:
m_me = HeckitRegression(method='mle').fit(
    y, X, Z,
    feature_names_outcome=['shared', 'x_only'],
    feature_names_selection=['shared', 'z_only'],
)
ame_cond = m_me.ame(kind='conditional')
print(ame_cond.summary())

                      Heckit Regression Marginal Effects                      
Dep. Variable:                      y
Method:                          dydx
At:                           overall
                   dy/dx    std err          z      P>|z|     [0.025     0.975]
------------------------------------------------------------------------------
shared            0.4086      0.020     20.771      0.000      0.370      0.447
x_only           -0.3017      0.019    -15.738      0.000     -0.339     -0.264
z_only           -0.2017      0.021     -9.568      0.000     -0.243     -0.160


All four marginal-effect kinds in a single table. Each row is averaged across the sample (AME); the entries are blank where the variable does not enter that equation.

In [16]:
kinds = ['latent', 'conditional', 'unconditional', 'prob-selected']
tbl = pd.DataFrame(index=sorted({n for k in kinds for n in m_me.ame(kind=k).names}))
for k in kinds:
    me = m_me.ame(kind=k)
    tbl[k] = pd.Series(me.margeff, index=me.names)
tbl.round(4)

,latent,conditional,unconditional,prob-selected
shared,0.4925,0.4086,0.3349,0.0859
x_only,-0.3017,-0.3017,-0.1527,NaN
z_only,NaN,-0.2017,0.2057,0.2064


Reading the columns:

- `latent` reproduces the slope coefficients $\beta$ (a sanity check).
- `conditional` differs from `latent` on `shared` (Z-side correction kicks in) and matches it exactly on `x_only` (X-only variable, no correction).
- `unconditional` is uniformly damped: $E[Y\cdot S]$ weighs the X-side by the selection probability $\Phi(Z'\gamma)$ rather than 1.
- `prob-selected` shows only the Z-side variables, scaled by $\phi(Z'\gamma)$.

**Cross-check via finite differences of `predict`.** The derivative formulas in `_heckit_effects.py` and the prediction code in `heckit.py` are written separately, so a central finite difference of `predict()` is an independent check of the analytic dy/dx. Below we evaluate at $X = \bar X$, $Z = \bar Z$ and perturb each regressor.

In [17]:
h = 1e-4
X_mean = X.mean(axis=0, keepdims=True)
Z_mean = Z.mean(axis=0, keepdims=True)
def cond_at(Xm, Zm):
    return float(m_me.predict(X=Xm, Z=Zm, kind='conditional').mean())

mem = m_me.mem(kind='conditional')
rows = []
for j, name in enumerate(['shared', 'x_only']):
    Xp, Xm_ = X_mean.copy(), X_mean.copy()
    Xp[0, j] += h; Xm_[0, j] -= h
    Zp, Zm_ = Z_mean.copy(), Z_mean.copy()
    if name == 'shared':
        Zp[0, 0] += h; Zm_[0, 0] -= h
    num = (cond_at(Xp, Zp) - cond_at(Xm_, Zm_)) / (2*h)
    rows.append((name, mem.margeff[mem.names.index(name)], num))
# z_only is Z-only.
Zp, Zm_ = Z_mean.copy(), Z_mean.copy()
Zp[0, 1] += h; Zm_[0, 1] -= h
num_z = (cond_at(X_mean, Zp) - cond_at(X_mean, Zm_)) / (2*h)
rows.append(('z_only', mem.margeff[mem.names.index('z_only')], num_z))
pd.DataFrame(rows, columns=['variable', 'analytic dy/dx', 'finite-difference']).round(8)

,variable,analytic dy/dx,finite-difference
0,shared,0.405509,0.405509
1,x_only,-0.301747,-0.301747
2,z_only,-0.208959,-0.208959


The two columns agree to 7-8 decimals: the analytic derivative formulas in `_heckit_effects.py` are consistent with the prediction formulas in `heckit.py`.

---
## Real-data demonstration: Mroz (1987)

The canonical Heckit application — Wooldridge's wage equation for married women. Of the 753 observations in Mroz's dataset, 428 women are in the labor force (`inlf = 1`) and have an observed `lwage`; the remaining 325 do not work and `lwage` is missing. Running OLS on the 428 working women answers a conditional question — *what determines wages among those who choose to work?* — whereas the latent question — *what would women earn if everyone worked?* — is what Heckit estimates.

Following Wooldridge (2010, Example 17.5):

- **Outcome equation:** `lwage ~ educ + exper + expersq`
- **Selection equation:** `inlf ~ educ + exper + expersq + nwifeinc + age + kidslt6 + kidsge6`

`nwifeinc`, `age`, `kidslt6`, and `kidsge6` serve as the exclusion restriction (they shift the *decision to work* but are not supposed to affect wages directly).

In [18]:
try:
    import wooldridge as woo
    mroz = woo.data('mroz')
except ImportError:
    raise ImportError('Install with: pip install wooldridge')

print(f'shape: {mroz.shape}')
print(f'in labor force (inlf=1): {(mroz["inlf"]==1).sum()}')
print(f'out of labor force:      {(mroz["inlf"]==0).sum()}')
print(f'lwage missing where inlf=0: {mroz.loc[mroz["inlf"]==0, "lwage"].isna().sum()}/'
      f'{(mroz["inlf"]==0).sum()}  (should be 100% — the canonical selection setup)')

shape: (753, 22)
in labor force (inlf=1): 428
out of labor force:      325
lwage missing where inlf=0: 325/325  (should be 100% — the canonical selection setup)


**Step 1 — naive OLS on the working subsample.** This is what one gets by ignoring the selection problem. We will compare its coefficient on `educ` to the Heckit-corrected version below.

In [19]:
import statsmodels.formula.api as smf

ols = smf.ols('lwage ~ educ + exper + expersq',
              data=mroz[mroz['inlf'] == 1]).fit()
print(ols.summary().tables[1])

                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.5220      0.199     -2.628      0.009      -0.912      -0.132
educ           0.1075      0.014      7.598      0.000       0.080       0.135
exper          0.0416      0.013      3.155      0.002       0.016       0.067
expersq       -0.0008      0.000     -2.063      0.040      -0.002   -3.82e-05


**Step 2 — Heckit two-step.** The selection equation includes the exclusion variables; the second stage adds the inverse Mills ratio to the wage equation.

In [20]:
from censtrunc import HeckitRegression

heck_ts = HeckitRegression.from_formula(
    outcome   = 'lwage ~ educ + exper + expersq',
    selection = 'inlf ~ educ + exper + expersq + nwifeinc + age + kidslt6 + kidsge6',
    data=mroz, method='twostep',
).fit()
print(heck_ts.summary())

                         Heckman Selection Regression                         
Method:                  twostep                  No. Observations:        753
Selected (S=1):          428                      Censored (S=0):          325
sigma_e:                 0.663629                 rho:                     0.048614
                     Outcome equation (Y* = X * beta + e)                     
------------------------------------------------------------------------------
                     coef     std err         z     P>|z|      [0.025    0.975]
------------------------------------------------------------------------------
const             -0.5781      0.3051   -1.8948    0.0581     -1.1761    0.0199
educ               0.1091      0.0155    7.0243    0.0000      0.0786    0.1395
exper              0.0439      0.0163    2.6980    0.0070      0.0120    0.0758
expersq           -0.0009      0.0004   -1.9567    0.0504     -0.0017    0.0000
------------------------------------------

**Step 3 — Joint MLE Heckit** (asymptotically efficient).

In [21]:
heck_ml = HeckitRegression.from_formula(
    outcome   = 'lwage ~ educ + exper + expersq',
    selection = 'inlf ~ educ + exper + expersq + nwifeinc + age + kidslt6 + kidsge6',
    data=mroz, method='mle',
).fit()
print(heck_ml.summary())

                         Heckman Selection Regression                         
Method:                  mle                      No. Observations:        753
Selected (S=1):          428                      Censored (S=0):          325
sigma_e:                 0.663631                 rho:                     0.048580
Log-Likelihood:          -832.8972
                     Outcome equation (Y* = X * beta + e)                     
------------------------------------------------------------------------------
                     coef     std err         z     P>|z|      [0.025    0.975]
------------------------------------------------------------------------------
const             -0.5781      0.2577   -2.2432    0.0249     -1.0832   -0.0730
educ               0.1091      0.0148    7.3617    0.0000      0.0800    0.1381
exper              0.0439      0.0148    2.9622    0.0031      0.0148    0.0729
expersq           -0.0009      0.0004   -2.0622    0.0392     -0.0017   -0.0000
-------

### Coefficient comparison

Lining up the three wage-equation slope estimates. Wooldridge's textbook reports an OLS return-to-education of about 0.108; the Heckit-corrected return is close to that, and the implied correlation $\hat\rho$ between the wage-equation error and the participation error is informative about how strong the selection channel is.

In [22]:
import numpy as np
comparison = pd.DataFrame({
    'OLS (selected)': np.asarray(ols.params),
    'Heckit two-step': heck_ts.coef_,
    'Heckit MLE':     heck_ml.coef_,
}, index=heck_ts.outcome_feature_names_)
comparison.round(4)

,OLS (selected),Heckit two-step,Heckit MLE
const,-0.5220,-0.5781,-0.5781
educ,0.1075,0.1091,0.1091
exper,0.0416,0.0439,0.0439
expersq,-0.0008,-0.0009,-0.0009


In [23]:
pd.DataFrame({
    'two-step':  [heck_ts.rho_, heck_ts.sigma_],
    'MLE':       [heck_ml.rho_, heck_ml.sigma_],
}, index=['rho', 'sigma']).round(4)

,two-step,MLE
rho,0.0486,0.0486
sigma,0.6636,0.6636


Interpreting $\hat\rho$: a positive (negative) value means that the unobserved factors that raise wages also raise (lower) the probability of working. If $\hat\rho$ is small or insignificant (a Wald or LR test on $\rho = 0$ would say so), the selection correction is empirically unnecessary and ordinary OLS is fine.

### Marginal effects on Mroz

For policy interpretation we usually want marginal effects on the *conditional* wage $E[\log\text{wage}\mid \text{works}]$ and on the *participation probability* $P(\text{works})$. The participation effects below use the standard probit-style derivative $\gamma_v\,\phi(Z'\gamma)$, averaged across the sample.

In [24]:
ame_cond = heck_ml.ame(kind='conditional')
ame_prob = heck_ml.ame(kind='prob-selected')
pd.concat([
    ame_cond.summary_frame()[['dy/dx', 'std err', 'P>|z|']].rename(
        columns={'dy/dx': 'cond E[lwage]', 'std err': 'SE', 'P>|z|': 'pval'}),
    ame_prob.summary_frame()[['dy/dx', 'std err', 'P>|z|']].rename(
        columns={'dy/dx': 'P(in LF)', 'std err': 'SE_p', 'P>|z|': 'pval_p'}),
], axis=1).round(4)

,cond E[lwage],SE,pval,P(in LF),SE_p,pval_p
educ,0.1067,0.0143,0.0000,0.0394,0.0073,0.0000
exper,0.0417,0.0131,0.0015,0.0371,0.0052,0.0000
expersq,-0.0008,0.0004,0.0361,-0.0006,0.0002,0.0013
nwifeinc,0.0002,0.0007,0.7418,-0.0037,0.0015,0.0116
age,0.0010,0.0028,0.7356,-0.0159,0.0024,0.0000
kidslt6,0.0156,0.0463,0.7352,-0.2611,0.0319,0.0000
kidsge6,-0.0006,0.0021,0.7535,0.0108,0.0131,0.4069


The `educ` row reads: one extra year of schooling raises *expected log-wage among workers* by the `cond E[lwage]` figure, *and separately* raises the probability of being in the labour force by the `P(in LF)` figure.

Z-only variables (`nwifeinc`, `age`, `kidslt6`, `kidsge6`) still appear in the `cond E[lwage]` column with very small, statistically-insignificant entries. That is **not** a blank: they do affect the conditional wage, but only *through the Mills-ratio correction* $-\gamma_v\,\rho\,\sigma\,\delta(Z'\gamma)$. On Mroz the estimated correlation $\hat\rho$ is close to zero (about $0.03$), so that channel is numerically tiny and the entries land around zero — exactly as one would expect when the selection correction is weak.